# Phase A — NLP Factor Discovery (Critical Path)

**Goal:** Extract ≥5 dominant themes from a 75–100-paper corpus using TF-IDF and LDA, then map each theme to an operationalisable country-level variable.  
**Success criterion:** ≥5 coherent topics, each mapped to a measurable proxy (feeds Phase B).  
**Fallback Tier 1:** TF-IDF + KMeans if LDA topics are unstable.  
**Fallback Tier 2:** Structured manual review of 25 papers if both NLP methods fail.

### Steps
1. Load corpus metadata CSV
2. Preprocess text (lemmatise, remove stopwords + domain stoplist)
3. TF-IDF vectorisation
4. LDA topic modelling (gensim) + coherence diagnostics
5. Visualise topics (pyLDAvis)
6. Map topics → operationalisable variables
7. Export theme–variable mapping table

In [ ]:
# Deterministic seeds — reproducibility is an ethical commitment
RANDOM_SEED = 42

import random, numpy as np
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [ ]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
import gensim
import gensim.corpora as corpora
from gensim.models import LdaModel, CoherenceModel
import pyLDAvis
import pyLDAvis.gensim_models
import matplotlib.pyplot as plt

# Download NLTK resources on first run
for resource in ['stopwords', 'wordnet', 'omw-1.4', 'punkt']:
    nltk.download(resource, quiet=True)

## 1. Load corpus

Expected file: `data/raw/corpus_metadata.csv`  
Required columns: `title`, `abstract` (and optionally `authors`, `year`, `doi`).

**How to build the corpus CSV:**  
- Search Scopus / Web of Science with terms like: `("food insecurity" OR "food security") AND ("post-harvest loss" OR "cereal" OR "financial access")`.  
- Export results as CSV.  
- Place file at `data/raw/corpus_metadata.csv`.

In [ ]:
CORPUS_PATH = "../data/raw/corpus_metadata.csv"

df = pd.read_csv(CORPUS_PATH)
print(f"Papers loaded: {len(df)}")
df.head(3)

In [ ]:
# Combine title + abstract as the text unit for each paper
df["text"] = (df["title"].fillna("") + " " + df["abstract"].fillna("")).str.strip()
print(f"Papers with non-empty text: {(df['text'].str.len() > 0).sum()}")

## 2. Preprocessing

In [ ]:
import re

# Domain-specific stopwords — terms that are frequent but uninformative for topic discovery
DOMAIN_STOPLIST = {
    "food", "security", "insecurity", "study", "paper", "result",
    "analysis", "data", "country", "countries", "level", "using",
    "based", "model", "method", "approach", "show", "find", "found"
}

STOP_WORDS = set(stopwords.words("english")) | DOMAIN_STOPLIST
lemmatizer = WordNetLemmatizer()

def preprocess(text: str) -> list[str]:
    text = re.sub(r"[^a-z\s]", "", text.lower())
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in STOP_WORDS and len(t) > 2]
    return tokens

df["tokens"] = df["text"].apply(preprocess)
print("Sample tokens:", df["tokens"].iloc[0][:15])

## 3. TF-IDF (also used as Tier-1 fallback input)

In [ ]:
tfidf = TfidfVectorizer(max_features=500, min_df=2, max_df=0.90)
tfidf_matrix = tfidf.fit_transform(df["text"])
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

## 4. LDA topic modelling + coherence diagnostics

We sweep over a range of K (number of topics) and pick the K with highest coherence score.

In [ ]:
# Build gensim dictionary and corpus
dictionary = corpora.Dictionary(df["tokens"])
dictionary.filter_extremes(no_below=2, no_above=0.9)
bow_corpus = [dictionary.doc2bow(tokens) for tokens in df["tokens"]]

print(f"Vocabulary size after filtering: {len(dictionary)}")
print(f"Documents in corpus: {len(bow_corpus)}")

In [ ]:
# Coherence sweep — try K = 4 to 12 topics
K_RANGE = range(4, 13)
coherence_scores = {}

for k in K_RANGE:
    lda = LdaModel(
        corpus=bow_corpus,
        id2word=dictionary,
        num_topics=k,
        random_state=RANDOM_SEED,
        passes=10,
        alpha="auto",
        per_word_topics=True
    )
    cm = CoherenceModel(model=lda, texts=df["tokens"].tolist(),
                        dictionary=dictionary, coherence="c_v")
    coherence_scores[k] = cm.get_coherence()
    print(f"  K={k:2d}  coherence={coherence_scores[k]:.4f}")

best_k = max(coherence_scores, key=coherence_scores.get)
print(f"\nBest K = {best_k}  (coherence = {coherence_scores[best_k]:.4f})")

In [ ]:
# Plot coherence curve
plt.figure(figsize=(8, 4))
plt.plot(list(coherence_scores.keys()), list(coherence_scores.values()), marker="o")
plt.axvline(best_k, color="red", linestyle="--", label=f"Best K={best_k}")
plt.xlabel("Number of topics (K)")
plt.ylabel("Coherence score (c_v)")
plt.title("LDA coherence by number of topics")
plt.legend()
plt.tight_layout()
plt.savefig("../outputs/figures/lda_coherence_curve.png", dpi=150)
plt.show()

In [ ]:
# Fit final LDA model with best K
lda_final = LdaModel(
    corpus=bow_corpus,
    id2word=dictionary,
    num_topics=best_k,
    random_state=RANDOM_SEED,
    passes=20,
    alpha="auto",
    per_word_topics=True
)

# Print top 10 words per topic
for idx, topic in lda_final.print_topics(num_words=10):
    print(f"Topic {idx}: {topic}")

## 5. Visualise topics (pyLDAvis)

In [ ]:
pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim_models.prepare(lda_final, bow_corpus, dictionary)
pyLDAvis.save_html(vis, "../outputs/figures/lda_topics_vis.html")
vis

## 6. Map topics → operationalisable variables

Fill in the table below after inspecting the topic words above.  
Each row = one LDA topic → one measurable country-level proxy (for Phase B).

| Topic # | Top words | Theme label | Country-level proxy | Dataset source |
|---------|-----------|-------------|--------------------|-----------------|
| 0 | ... | Post-harvest loss | FAO FLW loss rate (%) | FAO FLW Database |
| 1 | ... | Financial access | Account ownership (%) | Global Findex 2021 |
| 2 | ... | Production / fertiliser | Fertiliser efficiency (yield/kg) | FAOSTAT |
| 3 | ... | Climate | Temp anomaly, rainfall CV | World Bank WDI |
| 4 | ... | Infrastructure | LPI score | World Bank LPI |

> **Document every mapping here.** This table goes into the dissertation and is the primary defence against R3 (mapping bias risk).

## 7. Export mapping table

In [ ]:
# Update this dict after completing the mapping above
mapping = {
    "topic_id": [],
    "top_words": [],
    "theme_label": [],
    "proxy_variable": [],
    "dataset_source": []
}

mapping_df = pd.DataFrame(mapping)
mapping_df.to_csv("../data/processed/phase_A_theme_variable_mapping.csv", index=False)
print("Mapping exported.")
mapping_df

---
## Tier-1 Fallback: TF-IDF + KMeans

Run this section **only if** LDA coherence is below ~0.40 or topics are not interpretable.

In [ ]:
# ── TIER-1 FALLBACK (run only if LDA fails) ───────────────────────────────
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD

N_CLUSTERS = 6  # adjust as needed

svd = TruncatedSVD(n_components=50, random_state=RANDOM_SEED)
tfidf_reduced = svd.fit_transform(tfidf_matrix)

km = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_SEED, n_init=10)
km.fit(tfidf_reduced)

# Top terms per cluster
feature_names = tfidf.get_feature_names_out()
order_centroids = km.cluster_centers_.argsort()[:, ::-1]
for i in range(N_CLUSTERS):
    top_terms = [feature_names[ind] for ind in order_centroids[i, :10]]
    print(f"Cluster {i}: {', '.join(top_terms)}")